# 13 — Rangkuman Percobaan Adversarial (Paper 2)

**Untuk presentasi / PPT & untuk menyusun `paper2-adversarial.tex`.** Notebook ini
merangkai *urutan cerita* percobaan adversarial: dari fondasi Paper 1, dua sumbu
ketangguhan, empat varian model, tiga rezim serangan, hingga temuan & kesimpulan.

> **Prinsip kejujuran data:** semua angka berasal dari eksperimen nyata
> (`paper2_pipeline_meta.json` dari notebook 11, dan `paper2_eval_results.json` dari
> notebook 12). Bila berkas tersedia (lokal / diunduh dari S3), notebook memuatnya;
> bila tidak, dipakai nilai *fallback* yang identik dengan hasil tercatat sehingga
> notebook tetap jalan (mis. sebelum eksperimen dijalankan).

Jalankan sel berurutan dari atas ke bawah.

## 0. Setup & pemuatan hasil

In [ ]:
import importlib, sys, subprocess
need=[m for m in ('matplotlib','pandas','numpy') if importlib.util.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.grid':True,'grid.alpha':0.3})

# Cari folder hasil (lokal notebook 11/12, folder induk, atau folder out).
def find_json(name):
    cands=[name, os.path.join('paper2_eval_out',name), os.path.join('paper2_models',name),
           os.path.join('..',name), name]
    for p in cands:
        if os.path.exists(p):
            print('  loaded:', p); return json.load(open(p))
    print('  (fallback):', name); return None
print('siap.')

## 1. Latar: Dua Sumbu Ketangguhan NIDS

Paper 1 menutup **sumbu-1 (perpindahan jaringan / distribution shift)** dengan
SFM + few-shot. Paper 2 menambah **sumbu-2 (evasion adversarial)**. Pertanyaan inti:
*bisakah satu XGBoost ringan pada 9 fitur SFM tangguh di KEDUA sumbu sekaligus?*

In [ ]:
# Diagram dua sumbu ketangguhan (konsep).
fig, ax = plt.subplots(figsize=(6.4,4.6)); ax.set_aspect('equal')
ax.axhline(0,color='k',lw=0.8); ax.axvline(0,color='k',lw=0.8)
ax.set_xlim(-0.1,1.1); ax.set_ylim(-0.1,1.1)
ax.set_xlabel('Sumbu-1: generalisasi lintas-jaringan (few-shot)')
ax.set_ylabel('Sumbu-2: ketahanan evasion (adversarial)')
pts={'baseline':(0.15,0.2,'#999999'),'few-shot':(0.85,0.25,'#4C72B0'),
     'adv':(0.2,0.8,'#DD8452'),'few-shot+adv\n(target ideal)':(0.85,0.8,'#55A868')}
for name,(x,y,c) in pts.items():
    ax.scatter([x],[y],s=260,color=c,edgecolor='k',zorder=3)
    ax.annotate(name,(x,y),textcoords='offset points',xytext=(0,-28),ha='center',fontsize=9)
ax.set_title('Peta konseptual: target Paper 2 = kuadran kanan-atas')
plt.tight_layout(); plt.show()
print('Catatan: posisi ini KONSEPTUAL. Posisi nyata ditentukan angka di Bagian 4-5.')
print('=== SEL 1 (dua sumbu) SELESAI ===')

## 2. Empat Varian Model (notebook 11)

Untuk tiap arah (CIC→UNSW, UNSW→CIC), dilatih 4 varian pada 9 fitur SFM
(XGBoost biner, z-score per-dataset fit-train-only):
1. **baseline** — sumber clean.
2. **few-shot** — sumber + 1% label target.
3. **adv** — sumber + adversarial training (D_clean ∪ D_adv, rasio 20%, eps_train=0.1).
4. **few-shot+adv** — (sumber + 1% target) lalu adversarial training (usulan Paper 2).

In [ ]:
meta = find_json('paper2_pipeline_meta.json')
cfg = {'eps_train':0.1,'adv_ratio':0.20,'fewshot_frac':0.01,
       'variants':['baseline','few-shot','adv','few-shot+adv'],
       'xgboost':'max_depth=8, lr=0.1, n_estimators=200, subsample/colsample=0.8'}
if meta:
    cfg['eps_train']=meta.get('eps_train',cfg['eps_train'])
    cfg['adv_ratio']=meta.get('adv_ratio',cfg['adv_ratio'])
    cfg['fewshot_frac']=meta.get('fewshot_frac',cfg['fewshot_frac'])
print('Konfigurasi pelatihan:')
for k,v in cfg.items(): print(f'  {k}: {v}')
print('=== SEL 2 (konfigurasi varian) SELESAI ===')

## 3. Tiga Rezim Serangan (notebook 12)

Semua pada eps ∈ {0.05, 0.1, 0.2}, saliency via *central finite-difference* (h=0.01):
- **Unconstrained** — FGSM bebas di ruang z-score (batas atas daya serang; bisa flow mustahil).
- **Functional-preserving** — FGSM + proyeksi ke ruang valid protokol (non-neg; paket integer;
  bytes≥pkts; mean=bytes/pkts; monotonik add-only) — serangan yang benar-benar dapat dikirim.
- **Adaptive white-box** — saliency dari model yang diserang sendiri (skenario terburuk).

In [ ]:
regimes = pd.DataFrame([
  {'Rezim':'Unconstrained','Realistis?':'Tidak','Makna':'batas atas daya serang'},
  {'Rezim':'Functional-preserving','Realistis?':'Ya','Makna':'flow valid protokol, dapat dikirim'},
  {'Rezim':'Adaptive white-box','Realistis?':'Ya (terburuk)','Makna':'penyerang tahu pertahanan'},
])
import IPython.display as ipd; ipd.display(regimes)
print('=== SEL 3 (rezim serangan) SELESAI ===')

## 4. Hasil: Tabel MCC empat varian (clean + evasion)

Dimuat dari `paper2_eval_results.json` (notebook 12). Bila belum ada, memakai
*fallback* placeholder (NaN) — ganti setelah eksperimen dijalankan.

In [ ]:
res = find_json('paper2_eval_results.json')
if res and res.get('rows'):
    dfe = pd.DataFrame(res['rows'])
    cols = ['arah','model','clean_source','clean_target',
            'unconstrained_eps0.1','adaptive_functional_eps0.1']
    cols = [c for c in cols if c in dfe.columns]
    ipd.display(dfe[cols])
else:
    print('Hasil belum tersedia. Jalankan notebook 11 lalu 12 di SageMaker,')
    print('unduh paper2_eval_results.json (S3 unsw-far/paper2_eval/), taruh di folder ini.')
    dfe = pd.DataFrame(columns=['arah','model','clean_source','clean_target',
                                'unconstrained_eps0.1','adaptive_functional_eps0.1'])
print('=== SEL 4 (tabel hasil) SELESAI ===')

In [ ]:
# Grafik: generalisasi (clean_target) vs ketahanan (adaptive_functional_eps0.1) per varian.
if not dfe.empty and 'clean_target' in dfe.columns:
    for direction in dfe['arah'].unique():
        sub = dfe[dfe['arah']==direction]
        labels=sub['model'].tolist(); x=np.arange(len(labels)); w=0.38
        fig,ax=plt.subplots(figsize=(7,3.8))
        ax.bar(x-w/2, sub['clean_target'], w, label='clean (lintas-jaringan)', color='#4C72B0')
        if 'adaptive_functional_eps0.1' in sub.columns:
            ax.bar(x+w/2, sub['adaptive_functional_eps0.1'], w,
                   label='adaptive functional evasion (eps=0.1)', color='#C44E52')
        ax.axhline(0,color='k',lw=0.8); ax.set_xticks(x); ax.set_xticklabels(labels,rotation=15)
        ax.set_ylabel('MCC'); ax.set_ylim(-0.3,1.0)
        ax.set_title(f'{direction}: generalisasi vs ketahanan evasion'); ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
else:
    print('(lewati grafik; hasil belum tersedia)')
print('=== SEL 4b (grafik hasil) SELESAI ===')

## 5. Alur Cerita (untuk narasi paper & slide)

1. **Motivasi** — NIDS rapuh di dua sumbu: perpindahan jaringan & evasion.
2. **Fondasi** — SFM + few-shot menutup sumbu-1 (Paper 1).
3. **Pertanyaan** — bisakah 1 XGBoost ringan tangguh di kedua sumbu sekaligus?
4. **Desain** — 4 varian x 2 arah; serangan finite-diff FGSM.
5. **Realisme** — bedakan unconstrained vs functional-preserving; uji adaptive white-box.
6. **Temuan** — (diisi angka) apakah few-shot+adv menjaga generalisasi SEKALIGUS tahan evasion,
   atau ada trade-off / runtuh di adaptive (temuan jujur = batas adversarial training pd pohon).
7. **Arah lanjut** — pertahanan tahan-adaptive; integrasi adaptasi online (Paper 3).

In [ ]:
# Diagram alur cerita (7 langkah).
fig, ax = plt.subplots(figsize=(11,2.2)); ax.axis('off')
steps=['Motivasi\n2 sumbu','Fondasi\nSFM+few-shot','Pertanyaan\n1 model, 2 sumbu?',
       '4 varian\nx 2 arah','3 rezim\nserangan','Temuan\n(angka)','Arah lanjut\n(Paper 3)']
n=len(steps); x=np.linspace(0.02,0.98,n)
for i,(xi,s) in enumerate(zip(x,steps)):
    ax.add_patch(plt.Rectangle((xi-0.065,0.35),0.13,0.3,fc='#EAF0F7',ec='#4C72B0',lw=1.5))
    ax.text(xi,0.5,s,ha='center',va='center',fontsize=8)
    if i<n-1:
        ax.annotate('',xy=(x[i+1]-0.07,0.5),xytext=(xi+0.07,0.5),
                    arrowprops=dict(arrowstyle='->',color='#333',lw=1.3))
ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title('Alur cerita percobaan adversarial (Paper 2)',fontsize=11)
plt.tight_layout(); plt.show()
print('=== SEL 5 (alur cerita) SELESAI ===')

## 6. Rangkuman Temuan Utama (slide penutup)

*(Isi angka final setelah notebook 11/12 dijalankan.)*

- **few-shot** memulihkan generalisasi lintas-jaringan, tetapi (hipotesis) tak menaikkan ketahanan evasion.
- **adv** menaikkan ketahanan evasion *functional-preserving*, tetapi tak menutup celah lintas-jaringan.
- **few-shot+adv** = kandidat terbaik dua-sumbu; perlu dicek apakah bertahan di *adaptive white-box*.
- Serangan *unconstrained* melebih-lebihkan ancaman dibanding *functional-preserving* (lebih realistis).
- Metrik utama **MCC** (tahan class imbalance), konsisten dengan Paper 1.

*Semua angka dilaporkan apa adanya; penurunan di skenario terburuk adalah temuan yang berharga.*